# ROGII Current-Only Neural Stack Integration — R3

This CV-only notebook tests the one useful result from R2: a trajectory MLP
trained from scratch on current-competition wells. It does **not** use external
weights and it does not create a competition submission.

The control is the existing target-free PF/beam/GR residual stack. Five outer
folds are grouped by well. Inside every outer fold, the trajectory MLP uses a
separate grouped early-stopping split. Its prediction is converted to TVT and
added only as a small distance-decayed correction to the OOF stack:

\[
T^{candidate}_i=T^{stack}_i + 0.10e^{-d_i/2400}
\left(T^{neural}_i-T^{stack}_i\right).
\]

Promotion requires a material pooled gain, at least four winning outer folds,
a majority of winning wells, no meaningful 600/1200-ft regression, and no
worst-decile degradation. Passing authorizes a separate full-training
inference notebook; it does not authorize a leaderboard submission by itself.


In [ ]:
# Experiment control panel
import os

SEED = 20260721
N_SPLITS = 5
TRAIN_PF_PARTICLES = 120
TRAIN_PF_SEEDS = 8
N_JOBS = 4
LGB_ESTIMATORS = 800
STACK_SHRINK = 0.95
STACK_CLIP = 60.0

PAST_FT = 300
FUTURE_FT = 300
N_GRID = 60
CURRENT_STRIDE = 200
MAX_LOCAL_SLOPE = 0.60
MLP_EPOCHS = 25
MLP_PATIENCE = 5
BATCH_SIZE = 1024

PRIMARY_WEIGHT = 0.10
PRIMARY_TAU = 2400.0
HORIZONS = (300, 600, 1200, 2400, 4800)

LOCAL_DEBUG = not os.path.exists("/kaggle/input")
SMOKE_MODE = os.environ.get("ROGII_SMOKE", "0") == "1" or LOCAL_DEBUG
FORCE_CPU = os.environ.get("ROGII_FORCE_CPU", "0") == "1"
if SMOKE_MODE:
    N_SPLITS = 2
    TRAIN_PF_PARTICLES = 50
    TRAIN_PF_SEEDS = 2
    LGB_ESTIMATORS = 100
    CURRENT_STRIDE = 600
    MLP_EPOCHS = 2
    MLP_PATIENCE = 2
    BATCH_SIZE = 512


In [ ]:
"""
Particle-filter geosteering tracker (numpy, vectorised over particles).

State per particle: (pos, rate) where pos = TVT + Z  ("structural" position that
is ~constant when geology is flat), rate = d(pos)/dMD. At each eval step we
propagate with momentum, convert to TVT = pos - Z, score against the typewell
GR(TVT) curve via a Gaussian likelihood, resample, and take the weighted-mean TVT.

Predictions are made only for the eval rows (TVT_input is NaN). A seed ensemble
is combined with weights ∝ exp(loglik/scale). This is the core signal that gets
geosteering solutions to single-digit RMSE.
"""
import numpy as np

# tuned defaults (close to strong public solutions)
MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5


def _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z, ir, gs,
            n_particles=500, seed=0):
    rng = np.random.default_rng(seed)
    N = n_particles
    ls = last_tvt + last_Z
    pos = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N
    lo, hi = tw_tvt[0] - 100, tw_tvt[-1] + 100
    out = np.empty(len(md)); loglik = 0.0
    prev_md = md[0] - 1.0
    for i in range(len(md)):
        dm = max(md[i] - prev_md, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dm + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - z[i], lo, hi)
        pos = tvt_p + z[i]
        if np.isfinite(gr[i]):
            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d = (gr[i] - eg) / gs
            lk = np.maximum(np.exp(-0.5 * np.minimum(d * d, 600.0)), 1e-300)
            avg = float((w * lk).sum()); loglik += np.log(max(avg, 1e-300))
            w = w * lk; s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        neff = 1.0 / (w * w).sum()
        if neff < RESAMP * N:
            cum = np.cumsum(w); u0 = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = float(np.dot(w, pos - z[i]))
        prev_md = md[i]
    return out, loglik


def pf_predict(hw, tw, n_particles=500, n_seeds=48, scale=5.0):
    """Return a full-length TVT array (known rows kept, eval rows predicted)."""
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)

    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out

    last = kn.iloc[-1]
    last_tvt = float(last["TVT_input"]); last_Z = float(last["Z"])
    # GR noise sigma from known section
    tw_at_k = np.interp(kn["TVT_input"].to_numpy(), tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn["GR"].fillna(0).to_numpy() - tw_at_k), 10., 60.))
    # initial structural rate from last 30 known points
    tail = kn.tail(30)
    dt = np.diff(tail["TVT_input"].to_numpy()); dz = np.diff(tail["Z"].to_numpy())
    dmv = np.diff(tail["MD"].to_numpy()); m = dmv > 0
    ir = float(np.median((dt + dz)[m] / dmv[m])) if m.sum() >= 3 else 0.0

    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    md = ev["MD"].to_numpy(float); z = ev["Z"].to_numpy(float)
    gr = gr_all[ev.index.to_numpy()]
    # prepend last known so momentum starts from the right place
    md = np.concatenate([[float(last["MD"])], md])
    z = np.concatenate([[last_Z], z])
    gr = np.concatenate([[np.nan], gr])

    preds = np.empty((n_seeds, len(md))); liks = np.empty(n_seeds)
    for s in range(n_seeds):
        preds[s], liks[s] = _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z,
                                    ir, gs, n_particles, seed=s)
    liks -= liks.max()
    wts = np.exp(liks / scale); wts /= wts.sum()
    est = (wts[:, None] * preds).sum(0)[1:]        # drop the prepended known row
    out[ev.index.to_numpy()] = est
    return out


In [ ]:
"""Beam-search geosteering tracker (numba JIT). Complements the particle filter:
finds a low-cost monotone-ish path through the typewell GR(TVT) grid that matches
the horizontal GR signature. Ensembled over several stiffness configs."""
import numpy as np
from numba import njit
from scipy.signal import savgol_filter

# (beam_size, move_cost, err_scale, savgol_radius)
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1),
    (10, 14.0, 90.0, 5), (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
]


@njit(cache=False)
def _beam(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.0; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt:
                    continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni:
                        fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]:
                        cC[fnd] = tot; cP[fnd] = bi
                elif nc < MAX:
                    cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]:
                    mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]:
            best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1):
        path[s] = hI[s, b]; b = hP[s, b]
    return path


def beam_predict(hw, tw):
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out
    last_tvt = float(kn.iloc[-1]["TVT_input"])
    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    hgr = gr_all[ev.index.to_numpy()]
    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
    paths = []
    for BS, mc, es, r in BEAM_CONFIGS:
        if r > 0 and len(hgr) > max(3, 2 * r + 1):
            win = min(2 * r + 1, len(hgr) if len(hgr) % 2 == 1 else len(hgr) - 1)
            sgr = savgol_filter(hgr, win, min(2, win - 1))
        else:
            sgr = hgr.copy()
        paths.append(tw_tvt[_beam(sgr.astype(np.float64), tw_gr, si, BS, float(mc), float(es))])
    out[ev.index.to_numpy()] = np.stack(paths, 0).mean(0)
    return out


In [ ]:
def ps_index(h):
    return int(h['TVT_input'].notna().sum())

"""
Compute honest domain signals per well (post-PS rows) for the LightGBM residual
stack. Target-free (no hidden TVT used). Saves parquet for train and test.

Signals: PF path (delta), beam path (delta), multi-scale NCC vs typewell,
self-correlation vs the lateral's own known section, Q-3D tortuosity, relative
trajectory geometry, GR texture, typewell GR residuals at PF/anchor baselines.
Target: dTVT = TVT - last_known_TVT (train only).
"""
import sys, os, glob, time
import numpy as np
import pandas as pd
from joblib import Parallel, delayed






ANCH_OFF = [-20, -10, -5, 0, 5, 10, 20]


def multiscale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2 * hw + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1] if len(ktvt) else 0.0, np.float32), np.zeros(nh, np.float32)))
            continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().to_numpy(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().to_numpy(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32); M = len(sts)
        if M == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]]
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw, mode="edge")
        Hm = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]]
        Hn = (Hm - Hm.mean(1, keepdims=True)) / (Hm.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win
        best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best] + hw, 0, nk - 1)].astype(np.float32), score))
    return out


def tortuosity(x, y, z, md, win=50):
    d = np.stack([np.gradient(x), np.gradient(y), np.gradient(z)], 1)
    dm = np.gradient(md)[:, None] + 1e-9
    t = d / dm
    n = np.linalg.norm(t, axis=1, keepdims=True) + 1e-9
    u = t / n
    kappa = np.linalg.norm(np.gradient(u, axis=0), axis=1)   # curvature proxy
    return pd.Series(kappa).rolling(win, min_periods=1).sum().to_numpy(np.float32)


def build_well(path, is_train):
    wid = os.path.basename(path).split("__")[0]
    base = os.path.dirname(path)
    try:
        h = pd.read_csv(path)
        tw = pd.read_csv(f"{base}/{wid}__typewell.csv")
    except Exception:
        return None
    ps = ps_index(h)
    if ps < 10 or ps >= len(h) - 3:
        return None
    if is_train and "TVT" not in h.columns:
        return None
    tw_s = tw.sort_values("TVT"); tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)
    if len(tw_tvt) < 3:
        return None

    tps = float(h["TVT_input"].iloc[ps - 1])
    md = h["MD"].to_numpy(float); x = h["X"].to_numpy(float); y = h["Y"].to_numpy(float); z = h["Z"].to_numpy(float)
    gr_all = h["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    ev = np.arange(ps, len(h)); nev = len(ev)

    pf = pf_predict(h, tw, n_particles=TRAIN_PF_PARTICLES if is_train else TEST_PF_PARTICLES, n_seeds=TRAIN_PF_SEEDS if is_train else TEST_PF_SEEDS, scale=12.0)
    bm = beam_predict(h, tw)
    ktvt = h["TVT_input"].to_numpy(float)[:ps]
    kgr = gr_all[:ps]; hgr = gr_all[ps:]
    ncc = multiscale_ncc(kgr, ktvt, hgr)
    tort = tortuosity(x, y, z, md)

    md_ps, x_ps, y_ps, z_ps = md[ps - 1], x[ps - 1], y[ps - 1], z[ps - 1]
    gr_s = pd.Series(gr_all)
    f = {
        "well": wid, "id": [f"{wid}_{i}" for i in ev],
        "last_known_tvt": tps,
        "pf_d": (pf[ev] - tps).astype(np.float32),
        "beam_d": (bm[ev] - tps).astype(np.float32),
        "pf_vs_beam": (pf[ev] - bm[ev]).astype(np.float32),
        "ncc8_d": np.clip(ncc[0][0] - tps, -80, 80), "ncc8_s": ncc[0][1],
        "ncc15_d": np.clip(ncc[1][0] - tps, -80, 80), "ncc15_s": ncc[1][1],
        "ncc25_d": np.clip(ncc[2][0] - tps, -80, 80), "ncc25_s": ncc[2][1],
        "ncc_mean_d": np.clip((ncc[0][0] + ncc[1][0] + ncc[2][0]) / 3 - tps, -80, 80),
        "tort": tort[ev],
        "d_md": (md[ev] - md_ps).astype(np.float32),
        "d_z": (z[ev] - z_ps).astype(np.float32),
        "d_xy": np.hypot(x[ev] - x_ps, y[ev] - y_ps).astype(np.float32),
        "dz_dmd": (np.gradient(z) / (np.gradient(md) + 1e-9))[ev].astype(np.float32),
        "gr": gr_all[ev].astype(np.float32),
        "gr_m21": gr_s.rolling(21, center=True, min_periods=1).mean().to_numpy()[ev].astype(np.float32),
        "gr_s21": gr_s.rolling(21, center=True, min_periods=1).std().fillna(0).to_numpy()[ev].astype(np.float32),
        "gr_d1": gr_s.diff().fillna(0).to_numpy()[ev].astype(np.float32),
        "frac": (np.arange(nev) / max(nev - 1, 1)).astype(np.float32),
    }
    for o in ANCH_OFF:
        f[f"twres_pf{o}"] = (gr_all[ev] - np.interp(pf[ev] + o, tw_tvt, tw_gr)).astype(np.float32)
    if is_train:
        f["target"] = (h["TVT"].to_numpy(float)[ev] - tps).astype(np.float32)
    return pd.DataFrame(f)




In [ ]:
import copy
import gc
import glob
import json
import logging
import random
import sys
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

default_work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WORK = Path(os.environ.get("ROGII_OUTPUT_DIR", str(default_work)))
WORK.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("rogii-neural-stack-r3")


def find_data_root():
    candidates = [
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
        Path("data"),
    ]
    for root in candidates:
        if (root / "train").exists() and (root / "sample_submission.csv").exists():
            return root
    if Path("/kaggle/input").exists():
        hits = list(Path("/kaggle/input").glob("**/train/*__horizontal_well.csv"))
        if hits:
            return hits[0].parent.parent
    raise FileNotFoundError("Could not locate competition data")


DATA = find_data_root()
all_paths = sorted((DATA / "train").glob("*__horizontal_well.csv"))
if SMOKE_MODE:
    all_paths = all_paths[:40]
selected_wells = np.asarray([p.name.split("__")[0] for p in all_paths])
assert len(selected_wells) >= N_SPLITS * 5

# Recent Kaggle PyTorch images can be incompatible with an assigned P100.
CUDA_VISIBLE = torch.cuda.is_available()
CUDA_COMPATIBLE = False
GPU_STATUS = "CUDA unavailable"
if CUDA_VISIBLE:
    try:
        capability = torch.cuda.get_device_capability(0)
        device_sm = 10 * capability[0] + capability[1]
        supported = [
            int(a.split("_")[1]) for a in torch.cuda.get_arch_list()
            if a.startswith("sm_") and a.split("_")[1].isdigit()
        ]
        minimum_sm = min(supported) if supported else 999
        CUDA_COMPATIBLE = device_sm >= minimum_sm
        GPU_STATUS = (
            f"{torch.cuda.get_device_name(0)} sm_{device_sm}; "
            f"PyTorch minimum sm_{minimum_sm}; compatible={CUDA_COMPATIBLE}"
        )
    except Exception as exc:
        GPU_STATUS = f"probe failed: {type(exc).__name__}: {exc}"
DEVICE = torch.device("cuda" if CUDA_COMPATIBLE and not FORCE_CPU else "cpu")
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    torch.set_num_threads(min(8, os.cpu_count() or 4))
LGB_DEVICE = "gpu" if CUDA_VISIBLE and not FORCE_CPU and not LOCAL_DEBUG else "cpu"
log.info("data=%s wells=%d smoke=%s", DATA, len(selected_wells), SMOKE_MODE)
log.info("accelerator=%s torch_device=%s lgb_device=%s", GPU_STATUS, DEVICE, LGB_DEVICE)


def build_train_signals():
    t0 = time.time()
    frames = Parallel(
        n_jobs=N_JOBS,
        verbose=5,
        prefer="threads" if LOCAL_DEBUG else "processes",
    )(
        delayed(build_well)(str(path), True) for path in all_paths
    )
    frames = [f for f in frames if f is not None and len(f)]
    result = pd.concat(frames, ignore_index=True)
    log.info(
        "signals rows=%d wells=%d elapsed=%.1fs",
        len(result), result["well"].nunique(), time.time() - t0,
    )
    return result


train = build_train_signals()
selected_wells = np.asarray(sorted(train["well"].unique()))
DROP = {"well", "id", "target", "last_known_tvt"}
FEATURES = [c for c in train.columns if c not in DROP]
X_STACK = np.nan_to_num(
    train[FEATURES].to_numpy(np.float32), nan=0.0, posinf=0.0, neginf=0.0
)
Y_STACK = train["target"].to_numpy(np.float32)
ROW_GROUPS = train["well"].to_numpy()


# Geometry-window representation used by the current-only R2 scratch control.
PAST_GRID = np.linspace(-(PAST_FT - 1), 0.0, N_GRID)
FUTURE_GRID = np.linspace(1.0, FUTURE_FT, N_GRID)


def clean_curve(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    good = np.isfinite(x) & np.isfinite(y)
    x, y = x[good], y[good]
    order = np.argsort(x); x, y = x[order], y[order]
    unique = np.r_[True, np.diff(x) > 1e-8]
    return x[unique], y[unique]


def encode_window(past, future=None):
    past = np.asarray(past, float)
    coef = np.polyfit(PAST_GRID, past, 1)
    linear_past = np.polyval(coef, PAST_GRID)
    detrended = past - linear_past
    scale = max(float(np.std(detrended)), 0.50)
    level = detrended / scale
    d1 = np.gradient(level); d2 = np.gradient(d1)
    extras = np.array([
        np.clip(coef[0] / 0.10, -5.0, 5.0),
        np.log1p(scale), np.mean(np.abs(d1)), np.mean(np.abs(d2)),
    ])
    features = np.concatenate([level, d1, d2, extras]).astype(np.float32)
    linear_future = np.polyval(coef, FUTURE_GRID)
    if future is None:
        return features, linear_future, scale
    target = ((np.asarray(future, float) - linear_future) / scale).astype(np.float32)
    return features, target


def curve_windows(x, y):
    x, y = clean_curve(x, y)
    lo = int(np.ceil(x[0] + PAST_FT - 1))
    hi = int(np.floor(x[-1] - FUTURE_FT))
    xs, ys = [], []
    if hi < lo:
        return xs, ys
    for anchor in np.arange(lo, hi + 1, CURRENT_STRIDE):
        past = np.interp(anchor + PAST_GRID, x, y)
        future = np.interp(anchor + FUTURE_GRID, x, y)
        seq = np.r_[past, future]
        if np.all(np.isfinite(seq)) and np.quantile(np.abs(np.diff(seq)), 0.99) <= MAX_LOCAL_SLOPE:
            xx, yy = encode_window(past, future)
            xs.append(xx); ys.append(yy)
    return xs, ys


WELLS = {}
window_x, window_y, window_g = [], [], []
for number, path in enumerate(all_paths, 1):
    well = path.name.split("__")[0]
    if well not in set(selected_wells):
        continue
    df = pd.read_csv(path, usecols=["MD", "Z", "TVT", "TVT_input"])
    data = {c.lower(): df[c].to_numpy(float) for c in ["MD", "Z", "TVT", "TVT_input"]}
    WELLS[well] = data
    md_clean, u_clean = clean_curve(data["md"], data["tvt"] + data["z"])
    xx, yy = curve_windows(md_clean, u_clean)
    window_x.extend(xx); window_y.extend(yy); window_g.extend([well] * len(xx))
    if number % 100 == 0:
        log.info("geometry loaded %d/%d wells", number, len(all_paths))
X_GEOM = np.asarray(window_x, np.float32)
Y_GEOM = np.asarray(window_y, np.float32)
G_GEOM = np.asarray(window_g)
log.info("geometry windows=%s wells=%d", X_GEOM.shape, len(np.unique(G_GEOM)))
assert len(X_GEOM) > 100


class ResidualBlock(nn.Module):
    def __init__(self, width=256, dropout=0.08):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(width), nn.Linear(width, width * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(width * 2, width), nn.Dropout(dropout),
        )
    def forward(self, x):
        return x + self.net(x)


class SurfaceMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.input = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, 256), nn.GELU())
        self.blocks = nn.Sequential(*[ResidualBlock() for _ in range(4)])
        self.output = nn.Sequential(nn.LayerNorm(256), nn.Linear(256, N_GRID))
    def forward(self, x):
        return self.output(self.blocks(self.input(x)))


def sequence_loss(pred, target):
    level = torch.nn.functional.smooth_l1_loss(pred, target)
    pd1 = pred[:, 1:] - pred[:, :-1]
    td1 = target[:, 1:] - target[:, :-1]
    slope = torch.nn.functional.smooth_l1_loss(pd1, td1)
    curve = torch.nn.functional.smooth_l1_loss(pd1[:, 1:] - pd1[:, :-1], td1[:, 1:] - td1[:, :-1])
    return level + 0.20 * slope + 0.05 * curve


def data_loader(x, y, shuffle):
    ds = TensorDataset(torch.from_numpy(x.astype(np.float32)), torch.from_numpy(y.astype(np.float32)))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0, pin_memory=DEVICE.type == "cuda")


@torch.no_grad()
def neural_predict(model, x):
    model.eval()
    outputs = []
    dummy = np.zeros((len(x), N_GRID), np.float32)
    for xb, _ in data_loader(np.asarray(x, np.float32), dummy, False):
        outputs.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.vstack(outputs)


def train_geometry(x_train, y_train, x_valid, y_valid, tag):
    torch.manual_seed(SEED)
    model = SurfaceMLP(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MLP_EPOCHS)
    best_score, best_state, best_epoch, stale = np.inf, None, 0, 0
    for epoch in range(1, MLP_EPOCHS + 1):
        model.train(); losses = []
        for xb, yb in data_loader(x_train, y_train, True):
            xb = xb.to(DEVICE, non_blocking=True); yb = yb.to(DEVICE, non_blocking=True)
            if not SMOKE_MODE:
                xb = xb + 0.003 * torch.randn_like(xb)
            optimizer.zero_grad(set_to_none=True)
            loss = sequence_loss(model(xb), yb)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step(); losses.append(float(loss.detach().cpu()))
        scheduler.step()
        pred = neural_predict(model, x_valid)
        score = float(np.sqrt(np.mean((pred - y_valid) ** 2)))
        log.info("%s epoch=%02d train=%.5f valid=%.5f", tag, epoch, np.mean(losses), score)
        if score < best_score - 1e-5:
            best_score = score
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
            best_epoch = epoch; stale = 0
        else:
            stale += 1
            if stale >= MLP_PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best_epoch, best_score


@torch.no_grad()
def predict_next(model, past):
    features, linear_future, scale = encode_window(past)
    residual = neural_predict(model, features[None, :])[0]
    return linear_future + scale * residual


def recursive_forecast(model, past, deltas):
    deltas = np.asarray(deltas, float)
    state = np.asarray(past, float)
    all_delta, all_value, start = [], [], 0.0
    while start < float(np.max(deltas)):
        future = predict_next(model, state)
        all_delta.extend(start + FUTURE_GRID); all_value.extend(future)
        state = future; start += FUTURE_FT
    return np.interp(deltas, np.asarray(all_delta), np.asarray(all_value))


def geometry_delta_for_well(model, well):
    data = WELLS[well]
    visible = np.isfinite(data["tvt_input"])
    last = np.flatnonzero(visible)[-1]
    anchor_md = data["md"][last]
    u_input = data["tvt_input"] + data["z"]
    past = np.interp(anchor_md + PAST_GRID, data["md"][: last + 1], u_input[: last + 1])
    hidden = np.arange(last + 1, len(data["md"]))
    deltas = data["md"][hidden] - anchor_md
    pred_u = recursive_forecast(model, past, deltas)
    pred_tvt = pred_u - data["z"][hidden]
    return np.clip(pred_tvt - data["tvt_input"][last], -80.0, 80.0)


LGB_PARAMS = dict(
    objective="regression",
    n_estimators=LGB_ESTIMATORS,
    learning_rate=0.02,
    num_leaves=127,
    min_child_samples=100,
    subsample=0.80,
    subsample_freq=1,
    colsample_bytree=0.70,
    reg_lambda=5.0,
    reg_alpha=1.0,
    verbose=-1,
    n_jobs=-1,
    random_state=SEED,
    device_type=LGB_DEVICE,
)


stack_oof = np.full(len(train), np.nan, float)
geometry_oof = np.full(len(train), np.nan, float)
fold_rows = []
outer = GroupKFold(n_splits=N_SPLITS)
for fold, (well_fit_idx, well_valid_idx) in enumerate(
    outer.split(selected_wells, groups=selected_wells), 1
):
    t0 = time.time()
    fit_wells = selected_wells[well_fit_idx]
    valid_wells = selected_wells[well_valid_idx]
    fit_rows = np.flatnonzero(np.isin(ROW_GROUPS, fit_wells))
    valid_rows = np.flatnonzero(np.isin(ROW_GROUPS, valid_wells))

    params = dict(LGB_PARAMS); params["random_state"] = SEED + fold
    try:
        stack_model = lgb.LGBMRegressor(**params).fit(X_STACK[fit_rows], Y_STACK[fit_rows])
    except Exception as exc:
        log.warning("fold%d LightGBM GPU failed (%s); retrying CPU", fold, exc)
        params["device_type"] = "cpu"
        stack_model = lgb.LGBMRegressor(**params).fit(X_STACK[fit_rows], Y_STACK[fit_rows])
    stack_oof[valid_rows] = np.clip(
        stack_model.predict(X_STACK[valid_rows]) * STACK_SHRINK, -STACK_CLIP, STACK_CLIP
    )

    geom_mask = np.isin(G_GEOM, fit_wells)
    xg, yg, gg = X_GEOM[geom_mask], Y_GEOM[geom_mask], G_GEOM[geom_mask]
    inner = GroupShuffleSplit(n_splits=1, test_size=0.18, random_state=SEED + fold)
    inner_fit, inner_valid = next(inner.split(xg, yg, gg))
    geom_model, best_epoch, inner_score = train_geometry(
        xg[inner_fit], yg[inner_fit], xg[inner_valid], yg[inner_valid], f"fold{fold}_geometry"
    )
    for well in valid_wells:
        idx = train.index[train["well"].eq(well)].to_numpy()
        pred = geometry_delta_for_well(geom_model, str(well))
        if len(pred) != len(idx):
            raise RuntimeError(f"geometry row mismatch {well}: {len(pred)} != {len(idx)}")
        geometry_oof[idx] = pred
    fold_rows.append({
        "fold": fold,
        "fit_wells": len(fit_wells),
        "valid_wells": len(valid_wells),
        "geometry_best_epoch": best_epoch,
        "geometry_inner_rmse": inner_score,
        "seconds": time.time() - t0,
    })
    log.info("fold=%d complete elapsed=%.1fs", fold, time.time() - t0)
    del stack_model, geom_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

assert np.isfinite(stack_oof).all() and np.isfinite(geometry_oof).all()
distance = train["d_md"].to_numpy(float)
decay = np.exp(-np.maximum(distance, 0.0) / PRIMARY_TAU)
selector = 0.85 * train["pf_d"].to_numpy(float) + 0.15 * train["beam_d"].to_numpy(float)
CANDIDATES = {
    "stack_control": stack_oof,
    "geometry_neural": geometry_oof,
    "selector_control": selector,
    "blend_fixed_w005": stack_oof + 0.05 * (geometry_oof - stack_oof),
    "blend_fixed_w010": stack_oof + 0.10 * (geometry_oof - stack_oof),
    "blend_decay_w005": stack_oof + 0.05 * decay * (geometry_oof - stack_oof),
    "blend_decay_w010_primary": stack_oof + PRIMARY_WEIGHT * decay * (geometry_oof - stack_oof),
    "blend_decay_w020": stack_oof + 0.20 * decay * (geometry_oof - stack_oof),
}


def pooled_rmse(actual, pred):
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(pred)) ** 2)))


summary_rows, well_rows = [], []
horizon_defs = [(str(h), distance <= h) for h in HORIZONS] + [("all", np.ones(len(train), bool))]
for name, pred in CANDIDATES.items():
    for horizon, mask in horizon_defs:
        frame = pd.DataFrame({
            "well": ROW_GROUPS[mask],
            "fold": train.loc[mask, "well"].map({
                w: f for f, (_, vi) in enumerate(outer.split(selected_wells, groups=selected_wells), 1)
                for w in selected_wells[vi]
            }).to_numpy(),
            "err2": (Y_STACK[mask] - pred[mask]) ** 2,
        })
        grouped = frame.groupby(["fold", "well"], as_index=False)["err2"].agg(["sum", "count"]).reset_index()
        grouped["rmse"] = np.sqrt(grouped["sum"] / grouped["count"])
        for row in grouped.itertuples(index=False):
            well_rows.append({
                "candidate": name, "horizon": horizon, "fold": int(row.fold),
                "well": row.well, "rows": int(row.count), "sse": float(row.sum),
                "rmse": float(row.rmse),
            })
        summary_rows.append({
            "candidate": name,
            "horizon": horizon,
            "rows": int(mask.sum()),
            "wells": int(grouped["well"].nunique()),
            "pooled_rmse": pooled_rmse(Y_STACK[mask], pred[mask]),
            "mean_well_rmse": float(grouped["rmse"].mean()),
            "median_well_rmse": float(grouped["rmse"].median()),
            "p90_well_rmse": float(grouped["rmse"].quantile(0.90)),
        })

summary = pd.DataFrame(summary_rows)
well_metrics = pd.DataFrame(well_rows)
pd.DataFrame(fold_rows).to_csv(WORK / "training_folds.csv", index=False)
summary.to_csv(WORK / "cv_summary.csv", index=False)
well_metrics.to_csv(WORK / "cv_well_metrics.csv", index=False)


def metric(candidate, horizon, column="pooled_rmse"):
    row = summary[(summary["candidate"] == candidate) & (summary["horizon"] == str(horizon))]
    return float(row.iloc[0][column])


PRIMARY = "blend_decay_w010_primary"
all_primary = metric(PRIMARY, "all")
all_stack = metric("stack_control", "all")
wm = well_metrics[well_metrics["horizon"].eq("all")].pivot(
    index=["fold", "well"], columns="candidate", values="rmse"
)
well_win_rate = float((wm[PRIMARY] < wm["stack_control"]).mean())
fold_compare = []
for fold in sorted(train["well"].map({
    w: f for f, (_, vi) in enumerate(GroupKFold(n_splits=N_SPLITS).split(selected_wells, groups=selected_wells), 1)
    for w in selected_wells[vi]
}).dropna().unique()):
    fold = int(fold)
    mask = train["well"].map({
        w: f for f, (_, vi) in enumerate(GroupKFold(n_splits=N_SPLITS).split(selected_wells, groups=selected_wells), 1)
        for w in selected_wells[vi]
    }).to_numpy() == fold
    fold_compare.append({
        "fold": fold,
        "stack_rmse": pooled_rmse(Y_STACK[mask], CANDIDATES["stack_control"][mask]),
        "primary_rmse": pooled_rmse(Y_STACK[mask], CANDIDATES[PRIMARY][mask]),
    })
fold_compare = pd.DataFrame(fold_compare)
fold_compare["primary_wins"] = fold_compare["primary_rmse"] < fold_compare["stack_rmse"]
fold_compare.to_csv(WORK / "fold_comparison.csv", index=False)
fold_wins = int(fold_compare["primary_wins"].sum())

checks = {
    "pooled_gain_at_least_half_percent": all_primary <= 0.995 * all_stack,
    "absolute_gain_at_least_point10": all_stack - all_primary >= 0.10,
    "wins_majority_of_wells": well_win_rate >= 0.52,
    "wins_at_least_four_folds": fold_wins >= 4,
    "safe_at_600ft": metric(PRIMARY, 600) <= 1.002 * metric("stack_control", 600),
    "safe_at_1200ft": metric(PRIMARY, 1200) <= 1.002 * metric("stack_control", 1200),
    "worst_decile_safe": metric(PRIMARY, "all", "p90_well_rmse") <= metric("stack_control", "all", "p90_well_rmse") + 0.05,
}
gate = {
    "passed": bool(all(checks.values())),
    "primary_candidate": PRIMARY,
    "checks": {k: bool(v) for k, v in checks.items()},
    "metrics": {
        "stack_all_rmse": all_stack,
        "primary_all_rmse": all_primary,
        "absolute_gain": all_stack - all_primary,
        "relative_gain": 1.0 - all_primary / all_stack,
        "well_win_rate": well_win_rate,
        "fold_wins": fold_wins,
        "stack_600": metric("stack_control", 600),
        "primary_600": metric(PRIMARY, 600),
        "stack_1200": metric("stack_control", 1200),
        "primary_1200": metric(PRIMARY, 1200),
        "stack_p90": metric("stack_control", "all", "p90_well_rmse"),
        "primary_p90": metric(PRIMARY, "all", "p90_well_rmse"),
    },
    "next_step": (
        "Build full-training inference integration; still require submission audit."
        if all(checks.values()) else
        "Reject neural integration; keep current contact/PF submission unchanged."
    ),
}
(WORK / "quality_gate.json").write_text(json.dumps(gate, indent=2), encoding="utf-8")
log.info("QUALITY GATE\n%s", json.dumps(gate, indent=2))

fig, ax = plt.subplots(figsize=(10, 6))
for name in ["stack_control", "geometry_neural", PRIMARY, "blend_decay_w020"]:
    part = summary[(summary["candidate"] == name) & (summary["horizon"] != "all")].copy()
    part["h"] = part["horizon"].astype(int)
    part = part.sort_values("h")
    ax.plot(part["h"], part["pooled_rmse"], marker="o", label=name)
ax.set_xlabel("Horizon after visible cut (ft)"); ax.set_ylabel("Grouped OOF RMSE (ft)")
ax.set_title("R3 current-only neural correction vs existing stack")
ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
fig.savefig(WORK / "cv_neural_stack_horizons.png", dpi=160)
plt.close(fig)

# CV-only invariant: this experiment is incapable of consuming a submission slot.
assert not (WORK / "submission.csv").exists()
print(summary.to_string(index=False))
print("R3 complete. Gate passed:", gate["passed"])
